In [ ]:
import pandas as pd
from pathlib import Path
from src.config import TARGET_COL

processed_dir = Path("../data/processed").resolve()

# baseline: без новых фич
df_clean = pd.read_csv("/Users/mnchk/hseml-group-project-mnchik/data/interim/covertype_clean.csv")

# вариант с feature engineering
df_feat = pd.read_csv( "/Users/mnchk/hseml-group-project-mnchik/data/processed/covertype_features.csv")

df = df_clean      # для baseline

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

In [ ]:
from src.data.split_data import split_dataset


X_train, X_val, X_test, y_train, y_val, y_test = split_dataset(df)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from src.models.metrics import compute_classification_metrics, format_metrics

baseline_pipe = Pipeline(
    steps=[
        ("scaler", StandardScaler(with_mean=False)),  
        ("clf", LogisticRegression(
            max_iter=1000,
            multi_class="multinomial",
            n_jobs=-1,
            random_state=42,
        )),
    ]
)

baseline_pipe.fit(X_train, y_train)
y_val_pred = baseline_pipe.predict(X_val)
y_test_pred = baseline_pipe.predict(X_test)

baseline_val_metrics = format_metrics(
    compute_classification_metrics(y_val, y_val_pred)
)
baseline_test_metrics = format_metrics(
    compute_classification_metrics(y_test, y_test_pred)
)

baseline_val_metrics, baseline_test_metrics

/Users/mnchk/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/mnchk/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


({'accuracy': 0.7237, 'f1_macro': 0.5313, 'f1_weighted': 0.7139},
 {'accuracy': 0.7228, 'f1_macro': 0.5303, 'f1_weighted': 0.7132})

Baseline-модель

В качестве базовой модели была выбрана многоклассовая логистическая регрессия без дополнительного feature engineering. В качестве признаков использовались только исходные фичи датасета после базовой очистки (приведение типов, удаление дубликатов). Перед обучением числовые признаки были стандартизированы с помощью `StandardScaler`, а бинарные признаки использовались без изменений.

Модель обучалась в виде пайплайна `StandardScaler + LogisticRegression` с использованием мультиклассовой схемы и фиксированного значения `random_state = 42`. Оценка качества проводилась на валидационной и тестовой выборках, выделенных ранее.

Полученные метрики для baseline-модели:

- Validation: accuracy  0.724, F1‑macro  0.531, F1‑weighted  0.714  
- Test: accuracy  0.723, F1‑macro  0.530, F1‑weighted   0.713  

Значение accuracy около 72 % показывает, что даже линейная модель способна извлекать полезную информацию из признаков и существенно лучше случайного угадывания (это около 14 %). F1‑macro около 0.53 указывает на то, что качество по отдельным классам сильно различается: модель заметно лучше работает на наиболее частых типах лесного покрова и хуже распознаёт редкие классы. Поэтому в дальнейшем основное внимание уделяется моделям с более сложной нелинейной структурой (ансамбли деревьев, бустинг), которые потенциально лучше справляются с многомерными взаимодействиями признаков и классовым дисбалансом.

Сначала запустила просто, тут перепроверка что правильно модель записала в файлах

In [14]:
from src.models.baseline import run_baseline

metrics = run_baseline()
metrics

/Users/mnchk/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/mnchk/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


{'val': {'accuracy': 0.7237, 'f1_macro': 0.5313, 'f1_weighted': 0.7139},
 'test': {'accuracy': 0.7228, 'f1_macro': 0.5303, 'f1_weighted': 0.7132}}

Все хорошо, работает

(В чате видела что для этапа ср1 достаточнор baseline, следующее просто наработка на будущее)

In [10]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import LinearSVC


models = {
    "logreg": Pipeline([
        ("scaler", StandardScaler(with_mean=False)),
        ("clf", LogisticRegression(
            max_iter=1000,
            multi_class="multinomial",
            n_jobs=-1,
            random_state=42,
        )),
    ]),
    "knn": Pipeline([
        ("scaler", StandardScaler(with_mean=False)),
        ("clf", KNeighborsClassifier(n_neighbors=15)),
    ]),
    "linear_svc": Pipeline([
        ("scaler", StandardScaler(with_mean=False)),
        ("clf", LinearSVC(
            C=1.0,
            random_state=42,
        )),
    ]),
    "random_forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        n_jobs=-1,
        random_state=42,
    ),
    "gradient_boosting": GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=3,
        random_state=42,
    ),
}

In [11]:
import pandas as pd
from copy import deepcopy

from src.models.metrics import compute_classification_metrics, format_metrics

experiments = []

for name, model in models.items():
    print(f"-{name}-")
    clf = deepcopy(model)

    clf.fit(X_train, y_train)
    y_val_pred = clf.predict(X_val)
    y_test_pred = clf.predict(X_test)

    val_metrics = format_metrics(compute_classification_metrics(y_val, y_val_pred))
    test_metrics = format_metrics(compute_classification_metrics(y_test, y_test_pred))

    exp_row = {
        "model": name,
        "params": str(getattr(clf, "get_params", lambda: {})()),
        "val_accuracy": val_metrics["accuracy"],
        "val_f1_macro": val_metrics["f1_macro"],
        "val_f1_weighted": val_metrics["f1_weighted"],
        "test_accuracy": test_metrics["accuracy"],
        "test_f1_macro": test_metrics["f1_macro"],
        "test_f1_weighted": test_metrics["f1_weighted"],
    }
    experiments.append(exp_row)

experiments_df = pd.DataFrame(experiments)
experiments_df.sort_values(by="val_f1_macro", ascending=False, inplace=True)
experiments_df

-logreg-


/Users/mnchk/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/mnchk/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


-knn-
-linear_svc-
-random_forest-
-gradient_boosting-


,model,params,val_accuracy,val_f1_macro,val_f1_weighted,test_accuracy,test_f1_macro,test_f1_weighted
3,random_forest,"{'bootstrap': True, 'ccp_alpha': 0.0, 'class_w...",0.9491,0.9158,0.9488,0.9494,0.9171,0.9491
1,knn,"{'memory': None, 'steps': [('scaler', Standard...",0.8968,0.8246,0.8963,0.8998,0.8302,0.8993
4,gradient_boosting,"{'ccp_alpha': 0.0, 'criterion': 'friedman_mse'...",0.7906,0.7280,0.7881,0.7923,0.7268,0.7897
0,logreg,"{'memory': None, 'steps': [('scaler', Standard...",0.7237,0.5313,0.7139,0.7228,0.5303,0.7132
2,linear_svc,"{'memory': None, 'steps': [('scaler', Standard...",0.7122,0.4605,0.6963,0.7115,0.4564,0.6957


In [12]:
from pathlib import Path

exp_dir = Path("../experiments").resolve()
exp_dir.mkdir(parents=True, exist_ok=True)
experiments_df.to_csv(exp_dir / "experiments.csv", index=False)

In [13]:
from sklearn.model_selection import GridSearchCV

rf = RandomForestClassifier(
    n_estimators=200,
    n_jobs=-1,
    random_state=42,
)

param_grid = {
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
}

grid = GridSearchCV(
    rf,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=3,
    n_jobs=-1,
    verbose=1,
)

grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print("Best CV f1_macro:", grid.best_score_)

best_rf = grid.best_estimator_
y_val_pred = best_rf.predict(X_val)
y_test_pred = best_rf.predict(X_test)

best_rf_val = format_metrics(compute_classification_metrics(y_val, y_val_pred))
best_rf_test = format_metrics(compute_classification_metrics(y_test, y_test_pred))
best_rf_val, best_rf_test

Fitting 3 folds for each of 27 candidates, totalling 81 fits


KeyboardInterrupt: 